## 0. Environment check
Confirms the notebook is running on the LinuxONE server (`s390x`) and shows the Python version.

In [7]:
import platform, sys
print(platform.machine(), sys.version)

s390x 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]


## 1. Connect to PostgreSQL
Loads the database URL from the `.env` file, opens a connection, and registers the pgvector type.
**Expected output:** the Postgres 16 version line and `('0.6.0',)`.

In [8]:
# --- Imports ---
import os                                  # read environment variables and expand file paths
from dotenv import load_dotenv             # loads settings from the .env file
import psycopg                             # PostgreSQL driver for Python
from pgvector.psycopg import register_vector  # teaches psycopg how to handle vector columns

# --- Load the database URL from .env (keeps passwords out of the notebook) ---
load_dotenv(os.path.expanduser("~/bank-rag-guard/.env"))

# --- Open one connection to the database ---
# autocommit=True means every statement is saved immediately, with no manual commit
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)
register_vector(conn)  # lets us pass numpy arrays as vector values

# --- Sanity checks: confirm the server and the pgvector extension ---
print(conn.execute("SELECT version()").fetchone()[0])  # Postgres version
print(conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone())  # pgvector version

PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on s390x-ibm-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
('0.6.0',)


## 2. Test vector similarity search
Creates a temporary table with two 3-number vectors and finds the one nearest to a query vector.
**Expected output:** `(1,)`. This proves pgvector search works.

In [9]:
import numpy as np  # numpy arrays are the format pgvector expects

# --- Create a temporary table (it vanishes when the connection closes) ---
# "vector(3)" means each row stores a 3-number vector
conn.execute("CREATE TEMP TABLE t (id int, v vector(3))")

# --- Insert two vectors: row 1 points along the x axis, row 2 along the y axis ---
conn.execute("INSERT INTO t VALUES (1, %s), (2, %s)",
             (np.array([1, 0, 0]), np.array([0, 1, 0])))

# --- Find the row closest to the query vector [0.9, 0.1, 0] ---
# "<->" is pgvector's distance operator; ORDER BY distance LIMIT 1 gives the nearest match
print(conn.execute("SELECT id FROM t ORDER BY v <-> %s LIMIT 1",
                   (np.array([0.9, 0.1, 0]),)).fetchone())  # expected: (1,)

(1,)


## 3. Create the core tables
- `chunks`: document pieces with a clearance `tier` (1-3) and an embedding vector
- `users`: demo accounts with a role and tier
- `audit_log`: hash-chained log of every event (tamper-evident)

In [10]:
# Safe to re-run: every statement uses IF NOT EXISTS

# --- Enable the vector extension (already done earlier; harmless to repeat) ---
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")

# --- chunks: the document pieces we search over ---
# "tier" is the clearance level (1 = lowest access, 3 = most restricted)
# "embedding" holds the 768-number vector from the Gemini embedding model
conn.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    id         BIGSERIAL PRIMARY KEY,
    source     TEXT NOT NULL,                       -- dataset name, e.g. 'enron' or 'cuad'
    doc_id     TEXT NOT NULL,                       -- id of the original document
    content    TEXT NOT NULL,                       -- the chunk text itself
    tier       SMALLINT NOT NULL CHECK (tier BETWEEN 1 AND 3),
    embedding  vector(768),
    created_at TIMESTAMPTZ NOT NULL DEFAULT now()
)
""")

# --- users: demo accounts, each with a role and a clearance tier ---
conn.execute("""
CREATE TABLE IF NOT EXISTS users (
    username TEXT PRIMARY KEY,
    role     TEXT NOT NULL CHECK (role IN ('teller', 'compliance', 'auditor')),
    tier     SMALLINT NOT NULL CHECK (tier BETWEEN 1 AND 3)
)
""")

# --- audit_log: tamper-evident log; each row's hash covers the previous row's hash ---
conn.execute("""
CREATE TABLE IF NOT EXISTS audit_log (
    id        BIGSERIAL PRIMARY KEY,
    ts        TIMESTAMPTZ NOT NULL DEFAULT now(),
    actor     TEXT NOT NULL,                        -- who did it (user or 'attacker-agent')
    action    TEXT NOT NULL,                        -- e.g. 'query', 'retrieve', 'guardrail_update'
    details   JSONB NOT NULL,                       -- structured event data
    prev_hash TEXT NOT NULL,                        -- hash of the previous row
    hash      TEXT NOT NULL                         -- SHA-256 over prev_hash + this row's data
)
""")
print("Tables created")

Tables created


## 4. Add indexes
A tier index for fast clearance filtering, and an HNSW index for fast nearest-neighbour search.

In [11]:
# --- Fast tier filtering (every search filters on tier) ---
conn.execute("CREATE INDEX IF NOT EXISTS chunks_tier_idx ON chunks (tier)")

# --- Fast nearest-neighbour search using HNSW with cosine distance ---
conn.execute("""
CREATE INDEX IF NOT EXISTS chunks_embedding_idx
ON chunks USING hnsw (embedding vector_cosine_ops)
""")
print("Indexes created")

Indexes created


## 5. Verify the tables
Lists the tables in the database. **Expected output:** `['audit_log', 'chunks', 'users']`.

In [12]:
# --- List the tables in the public schema; expect audit_log, chunks, users ---
rows = conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public' ORDER BY table_name
""").fetchall()
print([r[0] for r in rows])

['audit_log', 'chunks', 'users']
